In [0]:
import re
import zipfile
from datetime import datetime

VOLUME_LANDING_DIR = "/Volumes/workspace/nba_files/landing"
CANONICAL_LANDING_DIR = "/Volumes/workspace/nba_files/landing"
ZIP_NAME_OVERRIDE = None

def _ls(path):
    return list(dbutils.fs.ls(path))

def _exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def _rm(path, recurse=False):
    try:
        dbutils.fs.rm(path, recurse)
        return True
    except Exception:
        return False

def _cp(src, dst, recurse=False):
    dbutils.fs.cp(src, dst, recurse)

def _print_dir(path, limit=50):
    items = _ls(path)
    for x in items[:limit]:
        print(f"{x.path}  size={x.size}")
    if len(items) > limit:
        print(f"... ({len(items)-limit} more)")

def _find_latest_zip(landing_dir):
    zips = [f for f in _ls(landing_dir) if f.name.lower().endswith(".zip")]
    if not zips:
        return None
    def parse_date(name):
        m = re.search(r"(\d{4}-\d{2}-\d{2})", name)
        if m:
            try:
                return datetime.strptime(m.group(1), "%Y-%m-%d")
            except Exception:
                return None
        return None
    zips_sorted = sorted(
        zips,
        key=lambda f: (parse_date(f.name) or datetime.fromtimestamp(f.modificationTime/1000.0), f.modificationTime),
        reverse=True,
    )
    return zips_sorted[0]

def _unzip_to_volume(zip_path, dest_dir):
    local_zip = "/tmp/_landing_upload.zip"
    dbutils.fs.cp(zip_path, f"file:{local_zip}", True)
    with zipfile.ZipFile(local_zip, "r") as z:
        z.extractall(dest_dir)

def _cleanup_ds_store(base_dir):
    removed = []
    stack = [base_dir]
    while stack:
        p = stack.pop()
        try:
            for f in _ls(p):
                if f.name == ".DS_Store":
                    if _rm(f.path, False):
                        removed.append(f.path)
                elif f.path.endswith("/"):
                    stack.append(f.path.rstrip("/"))
        except Exception:
            pass
    if removed:
        print("Removed .DS_Store files:")
        for r in removed:
            print(r)
    else:
        print("No .DS_Store files found (or none removed).")

def _fix_nested_data_landing(canonical_dir):
    nested_root = f"{canonical_dir}/data/landing"
    if not _exists(nested_root):
        print("No nested /data/landing found. Nothing to fix.")
        return

    print("Detected nested path:", nested_root)
    print("Merging nested contents into canonical landing dir:", canonical_dir)

    nested_items = _ls(nested_root)
    for item in nested_items:
        src = item.path.rstrip("/")
        dst = f"{canonical_dir}/{item.name.rstrip('/')}"
        print("Copying", src, "->", dst)
        _cp(src, dst, True)

    print("Removing nested /data directory tree")
    _rm(f"{canonical_dir}/data", True)

def _verify_games_path(canonical_dir):
    games_path = f"{canonical_dir}/balldontlie/games"
    if not _exists(games_path):
        raise RuntimeError(f"Expected games path not found: {games_path}")

    dt_dirs = [x for x in _ls(games_path) if x.name.startswith("dt=")]
    print("Found dt partitions:", len(dt_dirs))
    if dt_dirs:
        print("Sample dt partitions:")
        for x in sorted(dt_dirs, key=lambda y: y.name)[-10:]:
            print(x.path)
    return games_path

print("=== 0) Verify landing directory exists ===")
if not _exists(VOLUME_LANDING_DIR):
    raise RuntimeError(f"Landing dir not found: {VOLUME_LANDING_DIR}")
_print_dir(VOLUME_LANDING_DIR)

print("\n=== 1) Pick zip to unzip ===")
if ZIP_NAME_OVERRIDE:
    zip_path = f"{VOLUME_LANDING_DIR}/{ZIP_NAME_OVERRIDE}"
    if not _exists(zip_path):
        raise RuntimeError(f"ZIP_NAME_OVERRIDE not found at: {zip_path}")
else:
    latest = _find_latest_zip(VOLUME_LANDING_DIR)
    if latest is None:
        raise RuntimeError(f"No zip files found in: {VOLUME_LANDING_DIR}")
    zip_path = latest.path
print("Using zip:", zip_path)

print("\n=== 2) Unzip into canonical landing directory ===")
_unzip_to_volume(zip_path, CANONICAL_LANDING_DIR)
print("Unzip complete.")

print("\n=== 3) Fix nested data/landing prefix if present ===")
_fix_nested_data_landing(CANONICAL_LANDING_DIR)

print("\n=== 4) Remove .DS_Store artifacts ===")
_cleanup_ds_store(CANONICAL_LANDING_DIR)

print("\n=== 5) Verify canonical landing paths ===")
print("Canonical landing contents:")
_print_dir(CANONICAL_LANDING_DIR)

print("\n=== 6) Verify BallDontLie games landing partitions ===")
games_path = _verify_games_path(CANONICAL_LANDING_DIR)

print("\n=== 7) Preview read (Bronze-ready) ===")
df_preview = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(games_path)
)
print("Rows:", df_preview.count())
display(df_preview.select("id", "season", "status", "dt", "meta.dt").limit(20))


In [0]:
zip_path = "/Volumes/workspace/nba_files/landing/landing_2026-02-19.zip"
dest_dir = "/Volumes/workspace/nba_files/landing"

display(dbutils.fs.ls(dest_dir))


In [0]:
dbutils.widgets.text("ZIP_PATH", zip_path)
dbutils.widgets.text("DEST_DIR", dest_dir)


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/nba_files/landing"))


In [0]:
%sh
unzip -o "/Volumes/workspace/nba_files/landing/balldontlie_games_2026-02-19.zip" -d "/Volumes/workspace/nba_files/landing"



In [0]:
display(dbutils.fs.ls("/Volumes/workspace/nba_files/landing/balldontlie/games"))


In [0]:
path = "/Volumes/workspace/nba_files/landing/balldontlie/games"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(path)
)

display(df.groupBy("season").count().orderBy("season"))
